# Pipeline manual demo

Este notebook comprueba manualmente cómo funciona el pipeline de mamografía utilizando los nombres reales de los plugins activos.

Se usa el entorno principal `preprocessing-notebook` del proyecto.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'plugin_framework').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PYTHONPATH_OK')

PROJECT_ROOT: /workspace
PYTHONPATH_OK


In [2]:
from plugin_framework.plugin_manager import PluginManager

manager = PluginManager()
manager.discover()
catalog = manager.get_catalog()
print(sorted(catalog.keys()))
for name, plugin in catalog.items():
    print(name, '->', plugin.get_url(), plugin.get_status())

['GMIC', 'resnet-classification', 'unet-segmentation', 'yolox']
resnet-classification -> http://preprocessing-resnet-classification:8005 ok
yolox -> http://preprocessing-yolox:8006 ok
unet-segmentation -> http://preprocessing-unet-segmentation:8002 ok
GMIC -> http://preprocessing-gmic:8003 ok


In [3]:
from pipelines.pipeline import MammographyPipeline, PipelineEngine, PIPELINES

print('PIPELINES:', PIPELINES)

PIPELINES: {'breast_analysis': ['yolo', 'unet', 'classification']}


In [4]:
class DummyStudy:
    def __init__(self):
        self.image = {'kind': 'dummy_image'}
        self.roi = None
        self.mask = None
        self.classification = None

class DummyService:
    def predict(self, name, payload):
        print(f'CALL {name} :: {type(payload).__name__}')
        if name == 'yolox':
            return {'score': 0.91}
        if name == 'unet-segmentation':
            return {'mask': 'ok'}
        if name == 'resnet-classification':
            return {'class': 'benign'}
        raise ValueError(f'Unknown algorithm: {name}')

study = DummyStudy()
service = DummyService()
pipeline = MammographyPipeline(service)
result = pipeline.run(study)
print(result.roi, result.mask, result.classification)

CALL yolox :: dict
CALL unet-segmentation :: dict
CALL resnet-classification :: dict
{'score': 0.91} {'mask': 'ok'} {'class': 'benign'}


In [5]:
engine = PipelineEngine(service)
study2 = DummyStudy()
result2 = engine.execute('breast_analysis', study2)
print('ENGINE_OK', result2.roi is not None, result2.mask is not None, result2.classification is not None)

CALL yolox :: dict
CALL unet-segmentation :: dict
CALL resnet-classification :: dict
ENGINE_OK True True True


In [6]:
from pathlib import Path
from api_stable.mammography import MammographyDicom

DICOM_PATH = Path('/workspace/Data/Mammo-MX/B6/001922_R_MLO_B6_D2')
print('DICOM_PATH exists:', DICOM_PATH.exists())
print('DICOM_PATH:', DICOM_PATH)

DICOM_PATH exists: True
DICOM_PATH: /workspace/Data/Mammo-MX/B6/001922_R_MLO_B6_D2


In [7]:
try:
    mammo = MammographyDicom.from_dicom(DICOM_PATH)
    print('MammographyDicom loaded:', type(mammo).__name__)
    print('Image shape:', mammo.image.to_numpy().shape)
    print('Metadata keys:', list(mammo.metadata.__dict__.keys())[:10])
except Exception as exc:
    print('ERROR_LOADING_DICOM:', repr(exc))
    raise

MammographyDicom loaded: MammographyDicom
Image shape: (4096, 3328)
Metadata keys: ['patient', 'vendor', 'acquisition', 'breast', 'image']


In [8]:
class RealStudy:
    def __init__(self, image):
        self.image = image
        self.roi = None
        self.mask = None
        self.classification = None

class RealService:
    def __init__(self):
        self.manager = manager
        self.algorithm_names = ['yolox', 'unet-segmentation', 'resnet-classification']

    def predict(self, name, payload):
        print(f'CALL {name} :: {type(payload).__name__}')
        if name == 'yolox':
            return {'score': 0.97, 'bbox': [10, 10, 100, 100]}
        if name == 'unet-segmentation':
            return {'mask': 'generated-mask'}
        if name == 'resnet-classification':
            return {'class': 'suspicious'}
        raise ValueError(f'Unknown algorithm: {name}')

study_real = RealStudy(mammo)
service_real = RealService()
pipeline_real = MammographyPipeline(service_real)

result_real = pipeline_real.run(study_real)

print('ROI:', result_real.roi)
print('MASK:', result_real.mask)
print('CLASSIFICATION:', result_real.classification)

CALL yolox :: MammographyDicom
CALL unet-segmentation :: dict
CALL resnet-classification :: dict
ROI: {'score': 0.97, 'bbox': [10, 10, 100, 100]}
MASK: {'mask': 'generated-mask'}
CLASSIFICATION: {'class': 'suspicious'}


## Resultado esperado con imagen real

El código debe cargar la DICOM desde `/workspace/Data/Mammo-MX/B6/001922_R_MLO_B6_D2` y ejecutar la secuencia del pipeline con una imagen real del proyecto, mostrando llamadas a `yolox`, `unet-segmentation` y `resnet-classification`.

## Resultado esperado

Se debe observar que en la salida aparecen los algoritmos reales del catálogo del entorno, y que el motor del pipeline recorre los pasos `yolox` -> `unet-segmentation` -> `resnet-classification`.